# SciAgent-Bench: GPU experiments (Colab)

Runtime -> Change runtime type -> **GPU** (L4 or A100 recommended; T4 works with `PROFILE = "quick"`). Results are
written to Google Drive so an interrupted session can resume; finished steps are skipped.

In [ ]:
PROFILE = "quick"              # "kaggle-t4" settings also work on an L4/A100
REPO_URL = "https://github.com/SyedaRubbani/sciagent-bench.git"
from google.colab import drive
drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/sciagent_results"
import os, subprocess, sys
if not os.path.exists("/content/sciagent-bench"):
    !git clone -q {REPO_URL} /content/sciagent-bench
%cd /content/sciagent-bench
os.environ["SCIAGENT_VLLM_VENV"] = "/content/vllm-env"

In [ ]:
!nvidia-smi -L
!pip install -q -e ".[dev,surrogate,mcp,plot]" "transformers>=4.44" "peft>=0.12" accelerate bitsandbytes hypothesis

## Run the pipeline
Each cell is one resumable step. Outputs go to `RESULTS/<step>/`; failures are written to `ERROR.txt` and do not stop later independent steps.

### env tests
Environment record and the test suite (torch kernels on CUDA).

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps env tests

### verify
Reference solutions with uncertainty (GPU Monte-Carlo and matrix-free PCG).

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps verify

### gpu_bench
Kernel throughput.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps gpu_bench

### bench_cpu
Classical planners: 20 seeds, two budgets, bootstrap CIs and paired tests.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps bench_cpu

### surrogate
Surrogate: curated data, GPU training, distillation, int8.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps surrogate

### bench_llm
Open-model LLM planner benchmark (vLLM if it installs, transformers otherwise).

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps bench_llm

### traces
Teacher traces on training seeds 1000+ (disjoint from evaluation seeds).

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps traces

### train_student
LoRA distillation of the 0.5B student.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps train_student

### inference
Serving efficiency: teacher, base, distilled fp16 and 4-bit.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps inference

### bench_distill
Planning quality of the distilled model on held-out seeds.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps bench_distill

### report
Technical report and results bundle.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps report

## Read the report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/report/TECHNICAL_REPORT.md").read()))
print(open(f"{RESULTS}/experiments_status.json").read())

The bundle `results_bundle.zip` (next to the results folder) contains every raw result, trace, cassette and log. Download it and run `sciagent report results/cpu <unzipped>/results --out docs/TECHNICAL_REPORT.md` in the repo to merge GPU and CPU results into one report.